# Exercise 4

## Import packages

In [168]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.optim
import torch.utils.data
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR

## Task 1
(3 points)

Implement the training loop for one training epoch.
An epoch trains on the whole training dataset once.

In [ ]:
def train(model : torch.nn.Module, use_cuda : bool, train_loader : torch.utils.data.DataLoader, optimizer : torch.optim.Optimizer, epoch : int, log_interval : int):
    """
    Train one epoch
    
    model -- the neural network
    use_cuda -- true if GPU should be used
    train_loader -- data loader
    optimizer -- network optimizer
    epoch -- number of current epoch
    log_interval -- number of training steps between logs
    """
    if(use_cuda):
        model.cuda()
        
    # set the model to train mode
    model.train()
    
    # enumerate over the dataloader to get mini batches
    #       of images and ground truth labels
    # HINT: the builtin python function enumerate() also gives you indices
    for batch_idx, (data, tg_class) in enumerate(train_loader):

        if use_cuda:
            data = data.cuda()
            tg_class = tg_class.cuda()
        
        # set the optimizers gradients to zero
        optimizer.zero_grad()
        
        #  run the network
        out = model(data)
        
        # compute negative log likelihood loss
        l = -torch.log(out[tg_class])
        
        #  do backpropagation
        l.backward()
        
        # Optimize
        optimizer.step()
        
        # print current loss for every nth ("log_interval"th) iteration
        if(batch_idx % log_interval == 0):
            print("Epoch %3d, Batch %4d, b. loss %5f" % (epoch, batch_idx, l))
        

We already implemented the validation function for you (this is essentially validate() from the last exercise)

In [170]:
def validate(model : torch.nn.Module , use_cuda, test_loader):
    """
    Compute test metrics
    
    model -- the neural network
    use_cuda -- true if GPU should be used
    test_loader -- data loader
    """
    # create a 10x10 grid of subplots
    _, axis = plt.subplots(10, 10)
    
    # set model to evaluation mode
    model.eval()
    test_loss = 0
    correct = 0
    plotted = 0
    
    # disable gradients globally
    with torch.no_grad():
        for batch_idx, (data, target) in enumerate(test_loader):
            # for each batch
            if use_cuda:
                # transfer to GPU
                data = data.cuda()
                target = target.cuda()
            
            # run network and compute metrics
            output = model(data)
            test_loss += F.nll_loss(output, target, reduction='sum').item()  # sum up batch loss
            pred = output.argmax(dim=1, keepdim=True)  # get the index of the max log-probability
            
            img_correct = pred.eq(target.view_as(pred))
            correct += pred.eq(target.view_as(pred)).sum().item()
            
            # plot the first 100 images
            img_idx = 0
            data = data.cpu().numpy()
            
            while plotted < 100 and img_idx < data.shape[0]:
                # compute position of ith image in the grid
                y = plotted % 10
                x = plotted // 10
                
                # convert image tensor to numpy array and normalize to [0, 1]
                img = data[img_idx, 0]
                img = (img - np.min(img)) / (np.max(img) - np.min(img))
                
                # make wrongly predicted images red
                img = np.stack([img] * 3, 2)
                if img_correct[img_idx] == 0:
                    img[:, :, 1:] = 0.0
                
                # disable axis and show image
                axis[y][x].axis('off')
                axis[y][x].imshow(img)
                
                # show the predicted class next to each image
                axis[y][x].text(30, 25, pred[img_idx].item())
                
                plotted += 1
                img_idx += 1
            
    test_loss /= len(test_loader.dataset)

    # show results
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset)))
    plt.show()

## Task 2
(4 points)

Implement a five-layer fully connected neural network.
The dimensions (without batch size) should change like this: 784->200->100->60->30->10
Use log softmax to compute the class predictions.

Run the code at the end of the notebook to train and validate your implementation.

### Task 2.1
* sigmoid non-linear activation function
* note that the last layer does not need an activation function!

### Task 2.2
* add a new class "FCNet2"
* replace sigmoid with ReLU

### Task 2.3
* add a new class "FCNet3"
* add batch normalization to the first and third layers (note the difference between 1D/2D/3D versions)


**NOTE:** The perfomance should improve slightly with each step. However, due to the random weight initialization applied by PyTorch, your results may vary a bit between trainings.

In [171]:
class FCNet1(nn.Module):
    """
    Fully Connected Neural Network
    
    Five fully connected layers with sigmoid non-linearity
    
    Dimensions
    784->200->100->60->30->10
    """
    def __init__(self):
        super(FCNet1, self).__init__()

        # initialize network layers
        # HINT: take a look at "torch.nn" (imported as "nn")
        self.network = nn.Sequential(
            nn.Linear(784, 200),
            nn.Linear(200, 100),
            nn.Linear(100, 60),
            nn.Linear(60, 30),
            nn.Linear(30, 10)
        )
    

    def forward(self, x : torch.Tensor):
        #  reshape batch of images to batch of 1D vectors
        # print(x.shape)
        # print((x.shape[0:-2] + (x.shape[-1] * x.shape[-2],)))
        # x = x.reshape((x.shape[0:-2] + (x.shape[-1] * x.shape[-2],)))
        x = x.flatten(start_dim=1)
        
        #  run network layers
        y = self.network(x)
        
        # compute log softmax over the output
        # HINT: take a look at "torch.nn.functional" (imported as "F")
        output = F.softmax(y, dim = 1)
        
        return output

In [172]:
class FCNet2(nn.Module):
    def __init__(self):
        super(FCNet2, self).__init__()

    def forward(self, x):
        pass

In [173]:
class FCNet3(nn.Module):
    def __init__(self):
        super(FCNet3, self).__init__()
        
    def forward(self, x):
        pass

## Task 3
(3 points)

Implement a convolutional neural network, consisting of two convolutional and two fully connected layers.
This time, the dimensions (without batch size) should change like this: 1x28x28->32x26x26->64x12x12->128->10

### Task 3.1
* two convolutional layers (kernel size 3)
* two fully-connected layers
* ReLU activation function

### Task 3.2
* add batch normalization to first convolutional and first fully connected layer

### Task 3.3
* use max pooling instead of stride to reduce the dimensions to 64x12x12

In [174]:
class ConvNet1(nn.Module):
    """
    Convolutional Neural Network
    
    Two convolutional layers and two fully connected layers
    
    Dimensions:
    1x28x28->32x26x26->64x12x12->128->10
    """
    def __init__(self):
        super(ConvNet1, self).__init__()
        
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1),
            nn.Conv2d(32, 64, 3, 2)
        )

        self.ann_layers = nn.Sequential(
            nn.Linear(64*12*12, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

        # TODO: initialize network layers
        

    def forward(self, x : torch.Tensor):
        # TODO: run convolutional layers

        x = self.cnn_layers(x)
        
        # TODO: reshape batch of images to batch of 1D vectors
        
        x = torch.flatten(start_dim=1)
        
        # TODO: run fully connected layers
       
        output = self.ann_layers(x)

        # TODO: compute log softmax over the output

        output = torch.log_softmax(x, dim=1)      
        
        return output

In [175]:
class ConvNet2(nn.Module):
    """
    Convolutional Neural Network
    
    Two convolutional layers and two fully connected layers
    
    Dimensions:
    1x28x28->32x26x26->64x12x12->128->10
    """
    def __init__(self):
        super(ConvNet2, self).__init__()
        
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, 2),
        )

        self.ann_layers = nn.Sequential(
            nn.Linear(64*12*12, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

        # TODO: initialize network layers
        

    def forward(self, x : torch.Tensor):
        # TODO: run convolutional layers

        x = self.cnn_layers(x)
        
        # TODO: reshape batch of images to batch of 1D vectors
        
        x = torch.flatten(start_dim=1)
        
        # TODO: run fully connected layers
       
        output = self.ann_layers(x)

        # TODO: compute log softmax over the output

        output = torch.log_softmax(x, dim=1)      
        
        return output

In [176]:
class ConvNet3(nn.Module):
    """
    Convolutional Neural Network
    
    Two convolutional layers and two fully connected layers
    
    Dimensions:
    1x28x28->32x26x26->64x12x12->128->10
    """
    def __init__(self):
        super(ConvNet3, self).__init__()
        
        self.cnn_layers = nn.Sequential(
            nn.Conv2d(1, 32, 3, 1),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 64, 3, 1),
            nn.MaxPool2d(2)
        )

        self.ann_layers = nn.Sequential(
            nn.Linear(64*12*12, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

        # TODO: initialize network layers
        

    def forward(self, x : torch.Tensor):
        # TODO: run convolutional layers

        x = self.cnn_layers(x)
        
        # TODO: reshape batch of images to batch of 1D vectors
        
        x = torch.flatten(start_dim=1)
        
        # TODO: run fully connected layers
       
        output = self.ann_layers(x)

        # TODO: compute log softmax over the output

        output = torch.log_softmax(x, dim=1)      
        
        return output

In [177]:
# hyper parameters
batch_size = 64
test_batch_size = 1000
epochs = 10
lr = 1.0
gamma = 0.7
log_interval = 100

# use GPU if available
use_cuda = torch.cuda.is_available()
print(use_cuda)
kwargs = {'num_workers': 1, 'pin_memory': True} if use_cuda else {}

# initialize data loaders
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=True, download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])), batch_size=batch_size, shuffle=True, **kwargs)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST('../data', train=False, 
    transform=transforms.Compose([
        transforms.ToTensor(), 
        transforms.Normalize((0.1307,), (0.3081,))
    ])),
    batch_size=test_batch_size, shuffle=True, **kwargs)

model = FCNet1()
if use_cuda:
    model = model.cuda()

# initialize optimizer and scheduler
optimizer = optim.Adadelta(model.parameters(), lr=lr)
scheduler = StepLR(optimizer, step_size=1, gamma=gamma)

for epoch in range(1, epochs + 1):
    # train one epoch
    train(model, use_cuda, train_loader, optimizer, epoch, log_interval)
    
    # run on test dataset
    validate(model, use_cuda, test_loader)
    scheduler.step()
    
    
    torch.save(model.state_dict(), "models/mnist/checkpoint.pt")

False
tensor([[0.0902, 0.0987, 0.1102, 0.1004, 0.0930, 0.0979, 0.0884, 0.1074, 0.1073,
         0.1066],
        [0.0895, 0.1108, 0.0949, 0.1037, 0.0964, 0.1029, 0.0779, 0.0978, 0.1056,
         0.1205],
        [0.0879, 0.1139, 0.0985, 0.0947, 0.0857, 0.0984, 0.0849, 0.1112, 0.1086,
         0.1161],
        [0.0865, 0.1092, 0.1005, 0.1038, 0.0927, 0.1063, 0.0815, 0.1003, 0.1064,
         0.1129],
        [0.0916, 0.0989, 0.1013, 0.0968, 0.0951, 0.0872, 0.0882, 0.1147, 0.1092,
         0.1169],
        [0.0950, 0.1180, 0.0953, 0.0914, 0.0983, 0.1002, 0.0940, 0.0935, 0.1017,
         0.1128],
        [0.0904, 0.0997, 0.1085, 0.1000, 0.0953, 0.0972, 0.0854, 0.1061, 0.1040,
         0.1134],
        [0.1004, 0.1195, 0.0991, 0.0905, 0.0903, 0.0987, 0.0854, 0.1122, 0.0929,
         0.1111],
        [0.0872, 0.1079, 0.1046, 0.0950, 0.0916, 0.1077, 0.0871, 0.1046, 0.1050,
         0.1093],
        [0.0934, 0.1156, 0.1016, 0.0951, 0.0891, 0.1111, 0.0855, 0.1028, 0.1006,
         0.1052],
    

KeyboardInterrupt: 